<a href="https://colab.research.google.com/github/palarunava/machine-learning-courses/blob/main/udacity-intro-to-tf-for-dl/flowers_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import json
import numpy as np

In [ ]:
IMG_WIDTH = 150
IMG_HEIGHT = 150
BATCH_SIZE = 32
EPOCHS=10

In [ ]:
ds, ds_info = tfds.load(
  'tf_flowers',
  split='train',
  with_info=True,
  as_supervised=True
)

print(f'Number of samples: {ds.cardinality()}')

train_size = int(0.8 * ds.cardinality().numpy())

train_ds = ds.take(train_size)
valid_ds = ds.skip(train_size)

# print(f'Number of samples: {ds_info.splits['train'].num_examples}')
print(f'Number of training samples: {train_ds.cardinality()}')
print(f'Number of validation samples: {valid_ds.cardinality()}')

classes = ds_info.features['label'].names

def data_distribution(ds, classes=classes):
  # Extract all numerical labels from train_ds, ensuring each label is a 1D array
  all_labels = np.concatenate([label.numpy().reshape(-1) for image, label in ds])

  label_counts = np.bincount(all_labels)
  label_distribution = {classes[i]: count for i, count in enumerate(label_counts)}

  print("\nLabel distribution in training dataset:")
  for label_name, count in label_distribution.items():
      print(f"  {label_name}: {count} samples")

# Data distribution in training dataset
data_distribution(train_ds)

# Data distribution in validation dataset
data_distribution(valid_ds)

In [ ]:
def display_images(ds, num_images=50):
  random_subset = ds.shuffle(buffer_size=1000, seed=52).take(num_images)

  num_images_per_row = 5
  num_rows = num_images // num_images_per_row
  image_size = 3

  f, axarr = plt.subplots(num_rows, num_images_per_row, figsize=(image_size * num_images_per_row, image_size * num_rows))

  i = j = 0
  for image, label in random_subset:
    axarr[i,j].imshow(image)
    axarr[i,j].set_title(f'{classes[label.numpy()]}', fontsize=10)
    axarr[i,j].axis('off')
    j += 1
    if j == num_images_per_row:
      i += 1
      j = 0

# Display random images
display_images(train_ds, num_images=10)

In [ ]:
def augment_dataset(ds):
  # Create a dedicated augmentation pipeline
  data_augmentation = tf.keras.Sequential([
    # 1. Resize the image first to the target width and height
    tf.keras.layers.Resizing(IMG_HEIGHT, IMG_WIDTH),

    # 2. Match: rescale=1./255 (Uncomment if your resize_function doesn't do it)
    tf.keras.layers.Rescaling(1./255),

    # 3. Match: horizontal_flip=True (The old setup didn't do vertical flips)
    tf.keras.layers.RandomFlip("horizontal"),

    # 4. Match: rotation_range=45
    # Keras treats the factor as a percentage of 360 degrees.
    # 45 / 360 = 0.125 (Yields random rotations between -45 and +45 degrees)
    tf.keras.layers.RandomRotation(factor=0.125, fill_mode="nearest"),

    # 5. Match: zoom_range=0.5
    # Represents an upper and lower bound for zooming in/out by up to 50%
    # tf.keras.layers.RandomZoom(height_factor=0.5, width_factor=0.5, fill_mode="nearest"),
    # --- STRICT ZOOM-IN ONLY ---
    # Represents a random zoom-in factor sampled between -50% and 0%
    tf.keras.layers.RandomZoom(height_factor=(-0.5, 0.0), width_factor=(-0.5, 0.0), fill_mode="nearest"),

    # 6. Match: width_shift_range=.15 and height_shift_range=.15
    # The old code shifted positions; the new alternative is RandomTranslation.
    # 0.15 represents shifting up to 15% of the total height/width dimensions.
    # tf.keras.layers.RandomTranslation(height_factor=0.15, width_factor=0.15, fill_mode="nearest")
  ])

  AUTOTUNE = tf.data.AUTOTUNE

  # Complete optimized pipeline
  augmented_dataset = (
      ds
      # 1. CACHE first (saves raw, un-augmented images to RAM)
      .cache()

      # 2. SHUFFLE next (perfectly randomizes individual image order)
      .shuffle(buffer_size=ds.cardinality())

      # 3. MAP augmentation (applies random flips/rotations to the entire batch)
      .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)

      # 4. BATCH third (groups the shuffled images into chunks)
      .batch(BATCH_SIZE)

      # 5. PREFETCH last (keeps the GPU fed constantly)
      .prefetch(buffer_size=AUTOTUNE)
  )
  return augmented_dataset

In [ ]:
#@title [OPTIONAL] Check Image Augmentations
samples = 15
augmented_dataset = augment_dataset(train_ds.skip(50).take(1).repeat(samples))
classes = ds_info.features['label'].names

num_images_per_row = 5
num_rows = samples // num_images_per_row
image_size = 3

f, axarr = plt.subplots(num_rows, num_images_per_row, figsize=(image_size * num_images_per_row, image_size * num_rows))

i = j = 0
for image, label in augmented_dataset.unbatch():
  if num_rows > 1:
    axarr[i,j].imshow(image)
    axarr[i,j].set_title(f'{classes[label.numpy()]}', fontsize=10)
    axarr[i,j].axis('off')
    j += 1
    if j == num_images_per_row:
      i += 1
      j = 0
  else:
    axarr[j].imshow(image)
    axarr[j].set_title(f'{classes[label.numpy()]}', fontsize=10)
    axarr[j].axis('off')
    j += 1

In [ ]:
augmented_train_dataset = augment_dataset(train_ds)

# Display random augmented images
display_images(augmented_train_dataset.unbatch(), num_images=10)

In [ ]:
model = tf.keras.Sequential([
  tf.keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
  tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2, 2),

  tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2,2),

  tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2,2),

  tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2,2),

  tf.keras.layers.Dropout(0.4),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(512, activation='relu'),
  tf.keras.layers.Dense(5)
])

model.compile(
  optimizer='adam',
  loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
  metrics=['accuracy']
)

model.summary()

In [ ]:
def preprocess_dataset(ds):
  AUTOTUNE = tf.data.AUTOTUNE

  # 1. Use a clean, fast Python function for individual image transformation
  def batch_preprocessing(image, label):
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

  return (
    ds
    # A. Resize individual elements first so they are uniform
    .map(batch_preprocessing, num_parallel_calls=AUTOTUNE)

    # B. Batch them into clean 4D matrices
    .batch(BATCH_SIZE)

    # C. CACHE the completed batches to RAM (saves enormous CPU/GPU cycles)
    .cache()

    # D. Prefetch to keep your GPU fed
    .prefetch(buffer_size=AUTOTUNE)
  )

# Prepare your validation data cleanly
rescaled_valid_ds = preprocess_dataset(valid_ds)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

early_stopping = tf.keras.callbacks.EarlyStopping(
  monitor='val_loss',     # The metric to watch (validation loss is standard)
  patience=3,             # Number of epochs to wait for an improvement before stopping
  restore_best_weights=True # Automatically rolls back model weights to the best epoch
)

history = model.fit(
  augmented_train_dataset,
  validation_data=rescaled_valid_ds,
  epochs=EPOCHS,
  # callbacks=[early_stopping]
)

history_dict = history.history

# 3. Save it to a file
with open('training_history.json', 'w') as f:
    json.dump(history_dict, f, indent=4)

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()